## Required Packages

In [3]:
import pandas as pd
import numpy as np

## Selected Files

In [5]:
input_file_path = '/Users/john_1john_1/Documents/Legant_Lab/h2b_tracking_csv_for_john240416/ds210418_ctrl_rep1_cell1_240416.csv'
output_file_path = '/Users/john_1john_1/Documents/Legant_Lab/Filtered Data/testing11111.csv'
selected_dim = 2
selected_max_fs = 3

## Longest Track Length (in frames)

In [7]:
##%%writefile "/Users/john_1john_1/Dropbox/Legant Lab/Scripts/Trajectory_Span.py"
def calculate_trajectory_span(file_path):

    largest_span = 0  # Largest difference in frames for any trajectory
    
    particle_data = pd.read_csv(file_path)
     
    for trajectory in np.unique(particle_data['traj_idx']):  # Iterate through each unique trajectory
        particle_traj = particle_data[particle_data['traj_idx'] == trajectory]  # boolean selector

        frame_numbers = np.array(particle_traj['t'])
        traj_span = frame_numbers.max() - frame_numbers.min()
        largest_span = max(largest_span, traj_span)  # Update largest span if current one is larger
        
    return largest_span
    

In [8]:
calculate_trajectory_span(input_file_path)

196.0

## Largest Uncertainty (in nm) 

In [10]:
%%writefile "/Users/john_1john_1/Dropbox/Legant Lab/JGparticle/Max_Uncertainty.py"
def maximum_uncertainty(file_path,dim):
    particle_data = pd.read_csv(file_path)
    if dim == 2:
        max_unc = particle_data['xy_pstd'].max()
    elif dim == 3:
        uncertainties = [particle_data['xy_pstd'].max(),particle_data['z_pstd'].max()]
        max_unc = max(uncertainties)
    else:
        print('please enter a valid dimension')
    return max_unc

Overwriting /Users/john_1john_1/Dropbox/Legant Lab/JGparticle/Max_Uncertainty.py


In [11]:
maximum_uncertainty(input_file_path,2)

NameError: name 'maximum_uncertainty' is not defined

## Maximum Jump Distance

In [ ]:
##%%writefile "/Users/john_1john_1/Dropbox/Legant Lab/Scripts/Max_Jump_Distance.py"
def max_jump_distance(max_dt,dim,file_path):
    disp_lst = [] #displacement values
    dt_lst = [] #time interval
    
    particle_data = pd.read_csv(file_path)
     
    for trajectory in np.unique(particle_data['traj_idx']): # Iterate through each unique trajectory
        particle_traj = particle_data[particle_data['traj_idx'] == trajectory] #boolean selector

        xarr = np.array(particle_traj['x [nm]']) #x,y,z for p trajectory in each frame
        yarr = np.array(particle_traj['y [nm]'])
        zarr = np.array(particle_traj['z [nm]'])
    
        for dt in range(1, max_dt + 1): #1-5 inclusive
            xlocs0 = xarr[:-dt]
            xlocs1 = xarr[dt:]  #dt frames ahead
            ylocs0 = yarr[:-dt]
            ylocs1 = yarr[dt:]  #dt frames ahead
            zlocs0 = zarr[:-dt] 
            zlocs1 = zarr[dt:]  #dt frames ahead
            
            if dim == 2:
                displacement = np.sqrt((xlocs1-xlocs0)**2 + (ylocs1-ylocs0)**2)
            elif dim == 3:
                displacement = np.sqrt((xlocs1-xlocs0)**2 + (ylocs1-ylocs0)**2 + (zlocs1-zlocs0)**2)
            else:
                return "Please enter a valid Dimension"
            
            disp_lst.extend(displacement)

    return max(disp_lst)

In [ ]:
max_jump_distance(selected_max_fs,selected_dim,input_file_path)

## Convert Legant Data to be Compatible with TARDIS

In [ ]:
%%writefile "/Users/john_1john_1/Dropbox/Legant Lab/Scripts/Convert_Legant_CSV.py"

In [ ]:
%%writefile "/Users/john_1john_1/Dropbox/Legant Lab/Scripts/Convert_Legant_CSV.py"
def legant_compatible_TARDIS(input_file,output_file,dim):
    """Input file path, output file path, dimensions"""
    particle_data = pd.read_csv(input_file)
    if dim == 2:
        particle_data['z [nm]'] = 0
    elif dim != 3:
        raise ValueError("dim argument must be either 2 or 3.")
    thunderstorm_data = particle_data[['traj_idx','t','x [nm]', 'y [nm]', 'z [nm]']].copy()
    thunderstorm_data.columns = ['id','frame','x [nm]', 'y [nm]', 'z [nm]'] 
    # Save the filtered data to a CSV file
    thunderstorm_data.fillna(np.nan, inplace=True)

    # Save the processed data to a CSV file, specifying na_rep if you want 'NaN' in the CSV
    return thunderstorm_data.to_csv(output_file, index=False, na_rep='NaN')

In [ ]:
legant_compatible_TARDIS(input_file_path,output_file_path,selected_dim)

# Creating Plotting Script

In [ ]:
%%writefile "/Users/john_1john_1/Dropbox/Legant Lab/JGparticle/Graphical_Comparison.py"
def histogram_plotting(TARDIS_df, PYTHON_df,log_scale):
    max_dt = len(df1.columns)  # Assuming df1's structure for time intervals
    n_cols = int(np.ceil(np.sqrt(max_dt)))
    n_rows = int(np.ceil(max_dt / n_cols))
    
    fig, axs = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axs = axs.flatten()  # Flatten the array of axes for easy indexing
    
    for i, col_name in enumerate(df1.columns):  # Iterate through column names
        # Plot histograms for both datasets on the same subplot
        axs[i].hist(df1[col_name], bins=20, alpha=0.5, label='TARDIS', edgecolor='black', density=True)
        axs[i].hist(df2[col_name], bins=20, alpha=0.5, label='PYTHON', edgecolor='black', density=True)
        
        axs[i].set_xlabel('Displacement (nm)')
        axs[i].set_ylabel('Probability Density')
        axs[i].set_title(f'Jump Distances for {col_name}')
        axs[i].legend(loc='upper right')
    
    # Adjust layout to prevent overlap
    plt.tight_layout()
    return plt.show()